# Contextual versus isolated detail representations, by age

For each detail, the **contextualization shift** is `1 - cos(contextual vector, isolated vector)`: how differently GPT-OSS
represents the detail inside its text than when encoded alone. Both vectors are mean-pooled, at every stored layer.

This notebook follows the shift against

- **source age**: the age the story was written for
- **target age**: the age a summary was written for (younger, actual, older summaries of the same story)

for **original details** and for the **summary details** of each of the three summaries, and then relates it to the
span-search compression of each story. It complements `gpt_oss_detail_representation_alignment.ipynb`, which covers
the actual-age summary only, at the final layer.

Input: `contextualization_shift/details.csv` from `hpc/analysis/submit_span_search.sh`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import linregress, spearmanr, t as student_t

RESULTS = Path(os.environ.get("RESULTS_DIR", "results"))
SHIFT = RESULTS / "analysis/representation/detail_compression/contextualization_shift/details.csv"
SPAN = RESULTS / "analysis/representation/detail_compression/span_search/contextual_mean/summaries.csv"
LAYER = 12          # layer for the main figures (nearest stored layer is used)
AGE_BIN = 1.0
KINDS = ["original", "younger", "actual", "older"]     # details of the original text, then of each summary
COLORS = {"original": "#52514e", "younger": "#2a78d6", "actual": "#eb6834", "older": "#1baf7a"}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": MUTED,
    "axes.labelcolor": "#52514e", "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 110, "font.size": 10,
})

raw = pd.read_csv(SHIFT)
raw["kind"] = np.where(raw.record_type.eq("original"), "original", raw.condition)
raw = raw[raw.kind.isin(KINDS)]
layers = sorted(raw.layer.unique())
layer = layers[int(np.argmin(np.abs(np.asarray(layers) - LAYER)))]
print(f"stored layers {layers}; main layer {layer}")
print(raw.groupby("kind").agg(details=("shift", "size"), stories=("story_key", "nunique")))

In [ ]:
# One row per record and layer: the story is the unit, never the individual detail.
rec = (raw.groupby(["story_key", "kind", "record_id", "source_age", "target_age", "layer"], dropna=False)
          .agg(shift=("shift", "mean"), details=("shift", "size")).reset_index())
rec["target_age"] = rec.target_age.fillna(rec.source_age)
rec["age_gap"] = rec.target_age - rec.source_age         # how far the summary's target age is from the story's age
rec["age_bin"] = np.floor(rec.source_age / AGE_BIN) * AGE_BIN
main = rec[rec.layer.eq(layer)]

def binned(frame, y, x="age_bin", group="kind", n_boot=400, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for (name, bin_), g in frame.groupby([group, x]):
        values = g[y].to_numpy()
        boots = rng.choice(values, size=(n_boot, len(values))).mean(axis=1)
        rows.append({group: name, x: bin_, "mean": values.mean(), "lo": np.percentile(boots, 2.5), "hi": np.percentile(boots, 97.5), "n": len(values)})
    return pd.DataFrame(rows)

def slope(g, y="shift", x="source_age"):
    g = g.dropna(subset=[x, y])
    fit = linregress(g[x], g[y])
    half = student_t.ppf(0.975, len(g) - 2) * fit.stderr
    return fit.slope, fit.slope - half, fit.slope + half, fit.pvalue, len(g)

## 1. Shift versus source age, for original and summary details
Story-level mean shift at the main layer. Each line is one kind of text: the original, and the younger, actual and older summary of the same stories.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
for kind in KINDS:
    sub = main[main.kind.eq(kind)]
    if sub.empty:
        continue
    ax.scatter(sub.source_age, sub["shift"], s=7, alpha=0.15, color=COLORS[kind], linewidths=0)
    line = binned(sub, "shift").sort_values("age_bin")
    x = line.age_bin + AGE_BIN / 2
    ax.fill_between(x, line.lo, line.hi, color=COLORS[kind], alpha=0.13, linewidth=0)
    ax.plot(x, line["mean"], color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
ax.set_xlabel("Source age (years)"); ax.set_ylabel("Mean shift, 1 - cos(contextual, isolated)")
ax.set_title(f"Contextualization shift by source age (layer {layer})", loc="left", fontsize=11)
ax.legend(frameon=False, ncol=4, loc="upper right")
fig.tight_layout();

In [ ]:
rows = []
for kind in KINDS:
    g = main[main.kind.eq(kind)]
    if g.story_key.nunique() >= 4:
        s, lo, hi, p, n = slope(g)
        rows.append({"kind": kind, "stories": n, "slope per year": s, "ci low": lo, "ci high": hi, "p": p})
pd.DataFrame(rows).round(4)

## 2. Shift versus target age
For the summaries only: does asking for a younger or older reader change how context-dependent the summary's details are? Left, by summary type at each source age. Right, against the gap between target and source age.

In [ ]:
summ = main[main.kind.isin(["younger", "actual", "older"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
for kind in ["younger", "actual", "older"]:
    sub = summ[summ.kind.eq(kind)]
    line = binned(sub, "shift").sort_values("age_bin")
    x = line.age_bin + AGE_BIN / 2
    ax.fill_between(x, line.lo, line.hi, color=COLORS[kind], alpha=0.13, linewidth=0)
    ax.plot(x, line["mean"], color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
ax.set_xlabel("Source age (years)"); ax.set_ylabel("Mean shift of summary details"); ax.legend(frameon=False, title="summary")
ax = axes[1]
for kind in ["younger", "actual", "older"]:
    sub = summ[summ.kind.eq(kind)]
    ax.scatter(sub.age_gap + np.random.default_rng(1).normal(0, 0.05, len(sub)), sub["shift"], s=9, alpha=0.4, color=COLORS[kind], linewidths=0)
    ax.plot(sub.age_gap.mean(), sub["shift"].mean(), "o", color=COLORS[kind], mfc="#fcfcfb", mew=2, ms=9)
ax.set_xlabel("Target age minus source age (years)"); ax.set_ylabel("Mean shift of summary details")
fig.tight_layout();

## 3. Paired within story: summary details minus original details
Positive means the summary's details depend on their context more than the original's details do.

In [ ]:
wide = main.pivot_table(index=["story_key", "source_age"], columns="kind", values="shift").reset_index()
fig, ax = plt.subplots(figsize=(7.5, 4))
for kind in ["younger", "actual", "older"]:
    if kind not in wide:
        continue
    diff = (wide[kind] - wide["original"]).dropna()
    ages = wide.loc[diff.index, "source_age"]
    ax.scatter(ages, diff, s=9, alpha=0.3, color=COLORS[kind], linewidths=0)
    frame = pd.DataFrame({"kind": kind, "age_bin": np.floor(ages / AGE_BIN) * AGE_BIN, "d": diff})
    line = binned(frame, "d").sort_values("age_bin")
    ax.plot(line.age_bin + AGE_BIN / 2, line["mean"], color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=f"{kind} - original")
ax.axhline(0, color=MUTED, lw=1)
ax.set_xlabel("Source age (years)"); ax.set_ylabel("Difference in mean shift"); ax.legend(frameon=False)
fig.tight_layout();

## 4. Retained versus omitted original details
Which original details a summary keeps comes from the mpnet alignment (`retained_cosine`: a summary detail matches the original detail closely enough). The question here: are the details a summary **keeps** represented more or less differently in context than the ones it **drops**, and does that change with source age? Each summary type is its own panel, since a younger summary drops different details than an older one.

In [ ]:
RET_DIR = RESULTS / "analysis/natural_language/detail_alignment"
RETENTION_COLUMN = "retained_cosine"      # or retained_structured / retained_transport
GROUP_COLORS = {"retained": "#4a3aa7", "omitted": "#eda100", "central": "#008300", "peripheral": "#e34948"}

def load_retention(stories):
    merged = RET_DIR / "original_detail_retention.csv"
    paths = [merged] if merged.exists() else sorted(RET_DIR.glob("shard_*/original_detail_retention.csv"))
    if not paths:
        return None
    wanted = ["story_key", "condition", "summary_id", "orig_idx", RETENTION_COLUMN]
    ret = pd.concat([pd.read_csv(p, usecols=lambda c: c in wanted, low_memory=False) for p in paths], ignore_index=True)
    ret = ret[ret.story_key.isin(stories) & ret.condition.isin(["younger", "actual", "older"])].copy()
    if ret[RETENTION_COLUMN].dtype == object:
        ret[RETENTION_COLUMN] = ret[RETENTION_COLUMN].map({"True": True, "False": False})
    ret["retained"] = np.where(ret[RETENTION_COLUMN].astype(bool), "retained", "omitted")
    ret["summary_id"] = pd.to_numeric(ret.summary_id, errors="coerce")
    first = ret.groupby(["story_key", "condition"]).summary_id.transform("min")      # one summary per story and condition
    return ret[ret.summary_id.eq(first)][["story_key", "condition", "orig_idx", "retained"]]

def wslope(x, y, w):
    """Weighted OLS slope; weights are story resample counts."""
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    var = np.average((x - mx) ** 2, weights=w)
    return np.average((x - mx) * (y - my), weights=w) / var if var > 0 else np.nan

def slope_contrast(rows, group, a, b, n_boot=500, seed=0):
    """Age slope of the shift for group a, for group b, and their difference, with a story-level bootstrap interval."""
    stories = rows.story_key.unique()
    index = {s: i for i, s in enumerate(stories)}
    sid = rows.story_key.map(index).to_numpy()
    x, y = rows.source_age.to_numpy(float), rows["shift"].to_numpy(float)
    in_a, in_b = rows[group].eq(a).to_numpy(), rows[group].eq(b).to_numpy()
    def est(w):
        wa, wb = w[sid] * in_a, w[sid] * in_b
        return wslope(x[in_a], y[in_a], wa[in_a]), wslope(x[in_b], y[in_b], wb[in_b])
    base = est(np.ones(len(stories)))
    rng = np.random.default_rng(seed)
    boots = np.array([np.subtract(*est(np.bincount(rng.integers(0, len(stories), len(stories)), minlength=len(stories)).astype(float))) for _ in range(n_boot)])
    boots = boots[np.isfinite(boots)]
    return {"slope " + a: base[0], "slope " + b: base[1], "difference": base[0] - base[1],
            "diff ci low": np.percentile(boots, 2.5) if len(boots) else np.nan, "diff ci high": np.percentile(boots, 97.5) if len(boots) else np.nan,
            "stories": len(stories)}

originals = raw[raw.record_type.eq("original")].drop(columns=["condition", "summary_id", "target_age"], errors="ignore")   # the retention table supplies the summary condition
retention = load_retention(set(originals.story_key)) if "centrality" in raw else None
if retention is None:
    print(f"No retention table under {RET_DIR} (or no centrality column); sections 4 and 6 are skipped.")
else:
    joined = originals.merge(retention, left_on=["story_key", "detail_index"], right_on=["story_key", "orig_idx"])
    rr = (joined.groupby(["story_key", "condition", "source_age", "retained", "layer"])["shift"].mean().reset_index())
    rr = rr[rr.groupby(["story_key", "condition", "layer"]).retained.transform("nunique").eq(2)]
    rr["age_bin"] = np.floor(rr.source_age / AGE_BIN) * AGE_BIN
    rr_main = rr[rr.layer.eq(layer)]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.9), sharey=True)
    for ax, condition in zip(axes, ["younger", "actual", "older"]):
        data = rr_main[rr_main.condition.eq(condition)]
        for status in ["retained", "omitted"]:
            sub = data[data.retained.eq(status)]
            if sub.empty:
                continue
            ax.scatter(sub.source_age, sub["shift"], s=7, alpha=0.15, color=GROUP_COLORS[status], linewidths=0)
            line = binned(sub, "shift", group="retained").sort_values("age_bin")
            x = line.age_bin + AGE_BIN / 2
            ax.fill_between(x, line.lo, line.hi, color=GROUP_COLORS[status], alpha=0.14, linewidth=0)
            ax.plot(x, line["mean"], color=GROUP_COLORS[status], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=status)
        ax.set_title(f"{condition} summary", loc="left", fontsize=10); ax.set_xlabel("Source age (years)")
    axes[0].set_ylabel("Mean shift of original details"); axes[0].legend(frameon=False)
    fig.tight_layout()
    table = []
    for condition in ["younger", "actual", "older"]:
        data = rr_main[rr_main.condition.eq(condition)]
        if data.story_key.nunique() >= 6:
            table.append({"summary": condition, **slope_contrast(data, "retained", "retained", "omitted")})
    display(pd.DataFrame(table).round(4))

## 5. Central versus peripheral original details
Centrality is a detail's mean cosine to the other details of the same text, taken from the contextual vectors; a detail is **central** if it is at or above its story's median at the layer. Centrality and the shift both come from the contextual vectors, so an association between them is not independent evidence.

In [ ]:
if "centrality" not in raw:
    print("No centrality column in the shift table; rerun contextualization_shift.py from this branch.")
else:
    orig = originals.copy()
    median = orig.groupby(["story_key", "layer"])["centrality"].transform("median")
    orig["centrality_group"] = np.where(orig.centrality >= median, "central", "peripheral")
    cc = orig.groupby(["story_key", "source_age", "centrality_group", "layer"])["shift"].mean().reset_index()
    cc = cc[cc.groupby(["story_key", "layer"]).centrality_group.transform("nunique").eq(2)]
    cc["age_bin"] = np.floor(cc.source_age / AGE_BIN) * AGE_BIN
    cc_main = cc[cc.layer.eq(layer)]
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for group in ["central", "peripheral"]:
        sub = cc_main[cc_main.centrality_group.eq(group)]
        ax.scatter(sub.source_age, sub["shift"], s=7, alpha=0.15, color=GROUP_COLORS[group], linewidths=0)
        line = binned(sub, "shift", group="centrality_group").sort_values("age_bin")
        x = line.age_bin + AGE_BIN / 2
        ax.fill_between(x, line.lo, line.hi, color=GROUP_COLORS[group], alpha=0.14, linewidth=0)
        ax.plot(x, line["mean"], color=GROUP_COLORS[group], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=group)
    ax.set_xlabel("Source age (years)"); ax.set_ylabel("Mean shift of original details"); ax.legend(frameon=False)
    ax.set_title(f"Central versus peripheral details (layer {layer})", loc="left", fontsize=11)
    fig.tight_layout()
    display(pd.DataFrame([slope_contrast(cc_main, "centrality_group", "central", "peripheral")]).round(4))

## 6. How retention and centrality relate
If central details are kept more often, a retained-versus-omitted difference could just be a centrality difference. Each panel is one summary type: the share of original details kept, for central and peripheral details, by source age.

In [ ]:
if retention is not None and "centrality" in raw:
    both = orig.merge(retention, left_on=["story_key", "detail_index"], right_on=["story_key", "orig_idx"])
    both = both[both.layer.eq(layer)]
    both["kept"] = both.retained.eq("retained").astype(float)
    rate = both.groupby(["story_key", "condition", "source_age", "centrality_group"])["kept"].mean().reset_index()
    rate["age_bin"] = np.floor(rate.source_age / AGE_BIN) * AGE_BIN
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.9), sharey=True)
    for ax, condition in zip(axes, ["younger", "actual", "older"]):
        for group in ["central", "peripheral"]:
            sub = rate[rate.condition.eq(condition) & rate.centrality_group.eq(group)]
            if sub.empty:
                continue
            line = binned(sub, "kept", group="centrality_group").sort_values("age_bin")
            x = line.age_bin + AGE_BIN / 2
            ax.fill_between(x, line.lo, line.hi, color=GROUP_COLORS[group], alpha=0.14, linewidth=0)
            ax.plot(x, line["mean"], color=GROUP_COLORS[group], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=group)
        ax.set_title(f"{condition} summary", loc="left", fontsize=10); ax.set_xlabel("Source age (years)")
    axes[0].set_ylabel("Share of original details kept"); axes[0].legend(frameon=False)
    fig.tight_layout()

### 6.1 Retention and context-dependence across source age (GPT-OSS)
The continuous-age version of the MiniLM analysis (`contextual_vs_isolated_detail_compression.ipynb`, section 5.1).

**How the shift is calculated here.** For each original detail and each stored layer, one vector is read from GPT-OSS with the detail inside its whole text (the detail's tokens, mean-pooled at that layer), and one vector with the detail encoded alone. The shift is `1 - cosine` between the two, at the same layer. 0 means the surrounding text did not change how the model represents the detail; larger means it did. The main layer is set at the top of the notebook.

**What is plotted**, per summary type (younger, actual, older). Age is a continuous axis: one value per story, smoothed with a Gaussian kernel (width 0.75 years) and a story-resampling 95% band. All three measures compare details *within* a story:

- **AUC**: the chance that a *dropped* detail has a larger shift than a *kept* one (0.5 = no relation; above 0.5 = context-dependent details are dropped).
- **Retention ratio (log2)**: retention among the story's low-shift half over its high-shift half (0 = no relation).
- **Odds ratio** for being kept per +1 SD of shift, holding centrality, position and detail length fixed, sliding along age.

The shift grows with position in the text and varies with detail length, so the first two use the shift **adjusted** for position and length. "Kept" is the mpnet `retained_cosine` rule, so it is defined independently of GPT-OSS.

In [ ]:
import sys
for _here in [Path(os.environ.get("STORY_FEATURES_ROOT", ".")).resolve(), Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_here / "src/shared/retention_shift.py").exists():
        sys.path.insert(0, str(_here / "src"))
        break
from shared.retention_shift import adjust_shift, smooth_with_band, story_measures, windowed_odds

In [ ]:
def retention_frame(layer_value):
    o = originals[originals.layer.eq(layer_value)]
    f = o.merge(retention, left_on=["story_key", "detail_index"], right_on=["story_key", "orig_idx"])
    f["y"] = f.retained.eq("retained").astype(float)
    f["position_norm"] = f.detail_index / f.groupby(["story_key", "condition"]).detail_index.transform("max").clip(lower=1)
    f["centrality_pct"] = f.groupby(["story_key", "condition"]).centrality.rank(pct=True)
    f["detail_len"] = f.isolated_tokens.clip(lower=1) if "isolated_tokens" in f else 1.0
    f["log_len"] = np.log(f.detail_len)
    return f

if retention is not None:
    rf = retention_frame(layer)
    ages_all = rf.drop_duplicates("story_key")["source_age"]
    lo_age, hi_age = np.quantile(ages_all, [0.03, 0.97])
    GRID = np.round(np.arange(np.ceil(lo_age * 2) / 2, hi_age + 1e-9, 0.25), 2)
    BANDWIDTH = 0.75
    results, trends, odds_all = {}, [], {}
    for condition in ["younger", "actual", "older"]:
        sub = rf[rf.condition.eq(condition)].copy()
        sub["shift_adj"] = adjust_shift(sub)
        story_adj = story_measures(sub, shift="shift_adj")
        if len(story_adj) < 20:
            continue
        results[condition] = {
            "story": story_adj,
            "auc": smooth_with_band(story_adj, "auc", GRID, BANDWIDTH, seed=1),
            "rr": smooth_with_band(story_adj, "log2_rr", GRID, BANDWIDTH, seed=2),
        }
        odds_all[condition] = windowed_odds(sub, GRID[::2], BANDWIDTH, n_boot=60, seed=3, shift="shift_adj")
        for column, name in [("auc", "AUC"), ("log2_rr", "log2 retention ratio")]:
            x, y, w = (story_adj[c].to_numpy(float) for c in ["source_age", column, "weight"])
            fit = lambda i: np.polyfit(x[i], y[i], 1, w=np.sqrt(w[i]))[0]
            rng = np.random.default_rng(0)
            boots = [fit(rng.integers(0, len(x), len(x))) for _ in range(500)]
            trends.append({"summary": condition, "measure": name, "stories": len(story_adj), "change per year of age": fit(np.arange(len(x))),
                           "ci_lo": np.percentile(boots, 2.5), "ci_hi": np.percentile(boots, 97.5)})
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    panels = [("auc", 0.5, "AUC: dropped detail has the larger shift", "Dropped details are more context-dependent"),
              ("rr", 0.0, "log2 retention: low-shift half / high-shift half", "Low-shift details are kept more often")]
    for ax, (key, null, ylabel, title) in zip(axes[:2], panels):
        for condition, res in results.items():
            c = res[key]
            ax.fill_between(c.age, c.lo, c.hi, color=COLORS[condition], alpha=0.15, linewidth=0)
            ax.plot(c.age, c.estimate, color=COLORS[condition], linewidth=2.2, label=condition)
        ax.axhline(null, color=MUTED, linestyle="--", linewidth=1)
        ax.set_xlabel("Source age (years)"); ax.set_ylabel(ylabel); ax.set_title(title, loc="left", fontsize=10)
    axes[0].legend(frameon=False, title="summary")
    for condition, c in odds_all.items():
        axes[2].fill_between(c.age, c.lo, c.hi, color=COLORS[condition], alpha=0.15, linewidth=0)
        axes[2].plot(c.age, c.estimate, color=COLORS[condition], linewidth=2.2)
    axes[2].axhline(1, color=MUTED, linestyle="--", linewidth=1); axes[2].set_yscale("log")
    axes[2].set_xlabel("Source age (years)"); axes[2].set_ylabel("Odds ratio for being kept, per +1 SD of shift")
    axes[2].set_title("Odds ratio, sliding along age", loc="left", fontsize=10)
    for ax in axes:
        ax.plot(ages_all, np.full(len(ages_all), ax.get_ylim()[0]), "|", color="#555555", alpha=0.25, markersize=8)
        ax.set_xlim(GRID.min(), GRID.max())
    fig.tight_layout()
    display(pd.DataFrame(trends).round(4))
    print("A change per year whose interval excludes 0 means the link between context-dependence and being dropped changes with age.")
    print("The tick marks at the bottom show where the stories are; bands widen where there are few.")

## 7. Across layers
Left: average shift at each layer. Right: how steeply the shift changes with source age at each layer. Early layers carry little context, so large shifts there mean the two encodings differ in surface form (position, framing) rather than meaning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for kind in KINDS:
    g = rec[rec.kind.eq(kind)]
    by_layer = g.groupby("layer")["shift"].mean()
    axes[0].plot(by_layer.index, by_layer.values, color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
    slopes = {l: slope(gl)[0] for l, gl in g.groupby("layer") if gl.story_key.nunique() >= 4}
    axes[1].plot(list(slopes), list(slopes.values()), color=COLORS[kind], lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=kind)
axes[0].set_xlabel("Layer"); axes[0].set_ylabel("Mean shift"); axes[0].legend(frameon=False)
axes[1].axhline(0, color=MUTED, lw=1); axes[1].set_xlabel("Layer"); axes[1].set_ylabel("Change in shift per year of source age")
fig.tight_layout();

Slope differences by layer: **retained minus omitted** and **central minus peripheral**. A contrast that only shows at one layer is weak evidence.

In [ ]:
if retention is not None and "centrality" in raw:
    out = []
    for lay in layers:
        d = rr[rr.layer.eq(lay) & rr.condition.eq("actual")]
        if d.story_key.nunique() >= 6:
            out.append({"layer": lay, "contrast": "retained - omitted (actual summary)", **slope_contrast(d, "retained", "retained", "omitted", n_boot=200)})
        c = cc[cc.layer.eq(lay)]
        if c.story_key.nunique() >= 6:
            out.append({"layer": lay, "contrast": "central - peripheral", **slope_contrast(c, "centrality_group", "central", "peripheral", n_boot=200)})
    contrast = pd.DataFrame(out)
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for name, color in [("retained - omitted (actual summary)", GROUP_COLORS["retained"]), ("central - peripheral", GROUP_COLORS["central"])]:
        sub = contrast[contrast.contrast.eq(name)]
        ax.fill_between(sub.layer, sub["diff ci low"], sub["diff ci high"], color=color, alpha=0.15, linewidth=0)
        ax.plot(sub.layer, sub["difference"], color=color, lw=2, marker="o", ms=5, mfc="#fcfcfb", mew=1.5, label=name)
    ax.axhline(0, color=MUTED, lw=1); ax.set_xlabel("Layer"); ax.set_ylabel("Difference in age slope of the shift"); ax.legend(frameon=False)
    fig.tight_layout()

## 8. Relation to span-search compression
Per story and summary: does a larger contextualization shift go with each summary detail standing for more of the story? Both measures change with age, so the right panel compares the parts of each that age does not explain.

In [ ]:
if not SPAN.exists():
    print(f"Span-search results not found at {SPAN}; run the span search first.")
else:
    span = pd.read_csv(SPAN)
    span_layer = span.layer.unique()[int(np.argmin(np.abs(span.layer.unique() - layer)))]
    span = span[span.layer.eq(span_layer) & span.length_penalty.eq(span.length_penalty.unique()[np.argmin(np.abs(span.length_penalty.unique() - 0.1))])]
    link = (main[main.kind.isin(["younger", "actual", "older"])]
            .merge(span[["story_key", "condition", "mean_span_len"]], left_on=["story_key", "kind"], right_on=["story_key", "condition"]))
    print(f"{len(link)} story-summaries matched; span layer {span_layer}")
    def residual(y, x):
        return y - np.polyval(np.polyfit(x, y, 1), x)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for kind in ["younger", "actual", "older"]:
        g = link[link.kind.eq(kind)]
        axes[0].scatter(g["shift"], g.mean_span_len, s=12, alpha=0.6, color=COLORS[kind], linewidths=0, label=kind)
        axes[1].scatter(residual(g["shift"].to_numpy(), g.source_age.to_numpy()), residual(g.mean_span_len.to_numpy(), g.source_age.to_numpy()),
                        s=12, alpha=0.6, color=COLORS[kind], linewidths=0)
    rho = spearmanr(link["shift"], link.mean_span_len).statistic
    axes[0].set_title(f"Raw (Spearman {rho:.2f})", loc="left", fontsize=10)
    axes[0].set_xlabel("Mean contextualization shift"); axes[0].set_ylabel("Mean span length (source details per summary detail)"); axes[0].legend(frameon=False)
    rs, rl = residual(link["shift"].to_numpy(), link.source_age.to_numpy()), residual(link.mean_span_len.to_numpy(), link.source_age.to_numpy())
    axes[1].set_title(f"After removing the source-age trend (Spearman {spearmanr(rs, rl).statistic:.2f})", loc="left", fontsize=10)
    axes[1].set_xlabel("Shift, residual"); axes[1].set_ylabel("Span length, residual")
    fig.tight_layout()

## Reading these results

- Shift compares two different inputs (the detail in its text, and the detail alone), so part of it is always present. Look at how it changes with age and between summary types, not at its absolute size.
- The unit is the story or summary; individual details from one text are not independent.
- Younger summaries exist for few stories, so that line has the widest intervals.
- Mean pooling is used for both vectors, so the two are read out the same way.